<a href="https://colab.research.google.com/github/sahebnag/Indias-DPI-Engineering-Digital-Democracy-at-Billion-Person-Scale/blob/main/openai_gradio_design_generator_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Course-End Project: Creating Designs by Leveraging OpenAI and Gradio UI

This notebook builds a simple **AI design generation platform** using **OpenAI Image API** and **Gradio UI**.

The app accepts a text prompt and generates a custom campaign-style visual suitable for banners/posters, especially for Netflix-style digital marketing campaigns.

## 1. Install Required Libraries

Run this cell first in Google Colab.

In [1]:
!pip install -q openai gradio pillow requests

## 2. Import Libraries

We import Gradio for the UI, OpenAI for image generation, requests/io/PIL for image processing, and getpass to safely enter the API key.

In [2]:
import os
import io
import requests
from getpass import getpass
from PIL import Image
import gradio as gr
from openai import OpenAI

## 3. Configure OpenAI API Key

Enter your OpenAI API key when prompted. The key is not printed in the notebook output.

In [3]:
# Securely enter your OpenAI API key in Colab
os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

Enter your OpenAI API Key: ··········


## 4. Define the Image Generation Function

The `generate_image` function takes a user prompt, enhances it for marketing/banner/poster design, calls OpenAI's image generation API, downloads the generated image, and returns it for display in Gradio.

In [4]:
def generate_image(prompt, design_type, style, size):
    """
    Generate a campaign-style image using OpenAI Image API.

    Args:
        prompt (str): User's text description.
        design_type (str): Banner, poster, social media creative, etc.
        style (str): Visual style such as cinematic, minimalist, neon, etc.
        size (str): Image size supported by the model.

    Returns:
        PIL.Image.Image: Generated image.
    """
    if not prompt or not prompt.strip():
        raise gr.Error("Please enter a design prompt.")

    enhanced_prompt = f"""
    Create a high-quality {design_type} for a Netflix digital marketing campaign.
    Theme/idea: {prompt}
    Visual style: {style}
    Requirements:
    - cinematic composition
    - premium entertainment-brand look
    - strong focal subject
    - dramatic lighting
    - suitable for promotional banner or poster use
    - avoid using real Netflix logo unless explicitly provided by the user
    - leave clean space for marketing headline text
    """

    try:
        response = client.images.generate(
            model="gpt-image-1",
            prompt=enhanced_prompt,
            size=size,
            n=1
        )

        # gpt-image-1 may return base64 image data
        image_base64 = response.data[0].b64_json

        if image_base64:
            import base64
            image_bytes = base64.b64decode(image_base64)
            image = Image.open(io.BytesIO(image_bytes))
            return image

        # Fallback for URL-based responses, if model/account returns URL
        image_url = response.data[0].url
        image_response = requests.get(image_url, timeout=30)
        image_response.raise_for_status()
        return Image.open(io.BytesIO(image_response.content))

    except Exception as e:
        raise gr.Error(f"Image generation failed: {str(e)}")

## 5. Build the Gradio Interface

The interface includes:

- Textbox for entering the campaign prompt
- Dropdown for design type
- Dropdown for visual style
- Dropdown for image size
- Image output area for generated design

In [5]:
demo = gr.Interface(
    fn=generate_image,
    inputs=[
        gr.Textbox(
            label="Enter Design Prompt",
            placeholder="Example: A dark fantasy series poster showing a mysterious kingdom under red moonlight",
            lines=4
        ),
        gr.Dropdown(
            choices=["Poster", "Banner", "Social Media Creative", "Web Hero Image"],
            value="Poster",
            label="Design Type"
        ),
        gr.Dropdown(
            choices=[
                "Cinematic",
                "Dark thriller",
                "Sci-fi futuristic",
                "Minimalist premium",
                "Neon cyberpunk",
                "Fantasy epic",
                "Documentary realistic"
            ],
            value="Cinematic",
            label="Visual Style"
        ),
        gr.Dropdown(
            choices=["1024x1024", "1024x1536", "1536x1024"],
            value="1024x1024",
            label="Image Size"
        )
    ],
    outputs=gr.Image(type="pil", label="Generated Campaign Design"),
    title="AI Design Generator for Netflix Campaigns",
    description="Enter a creative prompt and generate customized banner/poster visuals using OpenAI and Gradio.",
    examples=[
        ["A suspense thriller poster showing a detective standing in heavy rain under red neon lights", "Poster", "Dark thriller", "1024x1536"],
        ["A futuristic space adventure banner with a lonely astronaut facing a glowing alien planet", "Banner", "Sci-fi futuristic", "1536x1024"],
        ["A royal fantasy drama poster with an ancient palace, golden clouds, and a powerful queen", "Poster", "Fantasy epic", "1024x1536"]
    ],
    allow_flagging="never"
)

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


## 6. Launch the Application

Run the cell below in Google Colab. Gradio will generate a public/shareable link.

In [6]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0bf560f3a8db624032.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 7. Project Result

This notebook demonstrates how OpenAI's image generation API can be combined with Gradio UI to create a web-based creative design platform. The platform converts text prompts into customized visual content for digital marketing campaigns.